[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuelcostac/generative-networks/blob/main/pretraining/huggingface_sentiment.ipynb)

This notebook is part of the [Generative Networks course](https://www.nahuelcosta.com/teaching/generative-networks).

# Adapting DistilBERT for Sentiment Classification

We close the chapter's motivating review-classification example by adapting a pretrained BERT-family encoder. We compare a frozen linear probe with full fine-tuning on exactly the same Rotten Tomatoes splits.

The model in this notebook is deliberately small. It demonstrates a pretraining objective and representation-transfer workflow; it is **not** a foundation model. Foundation models additionally require broad data, much larger capacity, and extensive compute.


This notebook uses a real pretrained model from the Hugging Face Hub. The classification head is task-specific and starts randomly initialized; the encoder already contains representations learned through large-scale language-model pretraining.

We will compare quality and cost, save the best fully fine-tuned checkpoint, reload it, and classify new reviews. LoRA and QLoRA remain in the separate reduction chapter.


## 0. Validated Hugging Face Dependencies


In [ ]:
from importlib import metadata
import subprocess
import sys

PINNED_PACKAGES = {"transformers": "5.14.1", "datasets": "5.0.0", "accelerate": "1.14.0"}
to_install = []
for distribution, expected in PINNED_PACKAGES.items():
    try:
        installed = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        to_install.append(f"{distribution}=={expected}")
if to_install:
    print("Installing the validated course environment:", ", ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *to_install])
for distribution, expected in PINNED_PACKAGES.items():
    actual = metadata.version(distribution)
    print(f"{distribution}: {actual}")
    assert actual == expected


## 1. Imports and Reproducibility


In [ ]:
import gc
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    Trainer, TrainingArguments, set_seed,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fast_dev_run = os.environ.get("FAST_DEV_RUN", "0") == "1"
print(f"Using device: {device}; FAST_DEV_RUN={fast_dev_run}")


## 2. Configuration


In [ ]:
MODEL_ID = "distilbert/distilbert-base-uncased"
DATASET_ID = "cornell-movie-review-data/rotten_tomatoes"
project_root = Path.cwd().parent if Path.cwd().name == "pretraining" else Path.cwd()
output_dir = project_root / "pretraining" / "outputs" / "huggingface_sentiment"
frozen_dir = output_dir / "frozen_probe"
finetuned_dir = output_dir / "full_finetune"
output_dir.mkdir(parents=True, exist_ok=True)

max_length = 128
frozen_epochs = 1 if fast_dev_run else 5
finetune_epochs = 1 if fast_dev_run else 2
batch_size = 8 if fast_dev_run else 16
max_steps = 2 if fast_dev_run else -1
print("Artifacts:", output_dir)


## 3. Rotten Tomatoes Sentiment Data

The published dataset contains 8,530 training, 1,066 validation, and 1,066 test sentences with binary negative/positive labels. Smoke mode takes deterministic 64-example subsets from each split.


In [ ]:
raw = load_dataset(DATASET_ID)
if fast_dev_run:
    raw = {name: split.shuffle(seed=SEED).select(range(min(64, len(split)))) for name, split in raw.items()}
print({name: len(split) for name, split in raw.items()})
print(raw["train"][0])

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_length)

tokenized = {name: split.map(tokenize, batched=True, remove_columns=["text"]) for name, split in raw.items()}
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if device.type == "cuda" else None)
print(tokenizer.convert_ids_to_tokens(tokenized["train"][0]["input_ids"][:20]))


## 4. Metrics and Comparable Training Configuration


In [ ]:
def classification_metrics(prediction):
    logits, labels = prediction
    predictions = np.argmax(logits, axis=-1)
    accuracy = float((predictions == labels).mean())
    tp = int(((predictions == 1) & (labels == 1)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())
    precision = tp / max(1, tp + fp); recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return {"accuracy": accuracy, "f1": f1}

def training_arguments(name, epochs, learning_rate):
    fast_strategy = "steps" if fast_dev_run else "epoch"
    return TrainingArguments(
        output_dir=str(output_dir / name), num_train_epochs=epochs, max_steps=max_steps,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size * 2,
        learning_rate=learning_rate, weight_decay=0.01, warmup_steps=0 if fast_dev_run else 100,
        eval_strategy=fast_strategy, save_strategy=fast_strategy,
        eval_steps=1 if fast_dev_run else None, save_steps=1 if fast_dev_run else None,
        load_best_model_at_end=True, metric_for_best_model="accuracy", greater_is_better=True,
        save_total_limit=1, logging_steps=1 if fast_dev_run else 50, report_to="none",
        fp16=device.type == "cuda", use_cpu=device.type != "cuda", seed=SEED, data_seed=SEED,
    )

def train_and_profile(model, name, epochs, learning_rate):
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    trainer = Trainer(
        model=model, args=training_arguments(name, epochs, learning_rate),
        train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
        data_collator=collator, processing_class=tokenizer, compute_metrics=classification_metrics,
    )
    start = time.perf_counter(); trainer.train(); elapsed = time.perf_counter() - start
    metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    return trainer, {
        "method": name, "accuracy": metrics["test_accuracy"], "f1": metrics["test_f1"],
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "seconds": elapsed,
        "peak_gpu_mb": torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else float("nan"),
    }


## 5. Frozen Encoder: Linear Probing

Only DistilBERT's pre-classifier and classifier are trained. Gradients cannot update the Transformer encoder, so this directly tests how linearly accessible sentiment is in its pretrained representation.


In [ ]:
set_seed(SEED)
frozen_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, id2label={0: "NEGATIVE", 1: "POSITIVE"}, label2id={"NEGATIVE": 0, "POSITIVE": 1},
)
for parameter in frozen_model.distilbert.parameters(): parameter.requires_grad_(False)
trainable_names = [name for name, parameter in frozen_model.named_parameters() if parameter.requires_grad]
assert trainable_names and all(name.startswith(("pre_classifier", "classifier")) for name in trainable_names)
print("Trainable tensors:", trainable_names)

frozen_trainer, frozen_result = train_and_profile(frozen_model, "frozen_probe_trainer", frozen_epochs, 1e-3)
frozen_trainer.save_model(frozen_dir); tokenizer.save_pretrained(frozen_dir)
print(frozen_result)


## 6. Full Fine-Tuning

We reload the identical pretrained checkpoint and update every parameter with a much smaller learning rate. This offers more adaptation capacity but stores gradients and optimizer state for the complete model.


In [ ]:
del frozen_trainer, frozen_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

set_seed(SEED)
full_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, id2label={0: "NEGATIVE", 1: "POSITIVE"}, label2id={"NEGATIVE": 0, "POSITIVE": 1},
)
assert all(parameter.requires_grad for parameter in full_model.parameters())
assert sum(p.numel() for p in full_model.parameters() if p.requires_grad) > frozen_result["trainable_parameters"]

full_trainer, full_result = train_and_profile(full_model, "full_finetune_trainer", finetune_epochs, 2e-5)
full_trainer.save_model(finetuned_dir); tokenizer.save_pretrained(finetuned_dir)
print(full_result)


## 7. Compare Quality and Cost


In [ ]:
results = [frozen_result, full_result]
for row in results:
    print(row)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
names = ["Frozen probe", "Full fine-tune"]
axes[0].bar(names, [row["accuracy"] for row in results]); axes[0].set_ylabel("Test accuracy"); axes[0].set_ylim(0, 1)
axes[1].bar(names, [row["f1"] for row in results]); axes[1].set_ylabel("Test F1"); axes[1].set_ylim(0, 1)
axes[2].bar(names, [row["trainable_parameters"] / 1e6 for row in results]); axes[2].set_ylabel("Trainable parameters (M)")
for axis in axes: axis.tick_params(axis="x", rotation=15); axis.grid(axis="y", alpha=0.3)
plt.tight_layout()


## 8. Reload the Best Fine-Tuned Model


In [ ]:
reloaded = AutoModelForSequenceClassification.from_pretrained(finetuned_dir).to(device)
reloaded.eval(); full_model.to(device).eval()
check_inputs = tokenizer("The performances make this film worth watching.", return_tensors="pt", truncation=True, max_length=max_length).to(device)
with torch.no_grad():
    live_logits = full_model(**check_inputs).logits
    reloaded_logits = reloaded(**check_inputs).logits
assert torch.equal(live_logits, reloaded_logits)
print("Reloaded logits match exactly.")


## 9. Inference on New Reviews


In [ ]:
reviews = [
    "A thoughtful story with excellent performances.",
    "The plot is tedious and the jokes never land.",
    "Beautifully filmed, although the ending feels rushed.",
]
inputs = tokenizer(reviews, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
with torch.no_grad(): probabilities = reloaded(**inputs).logits.softmax(-1).cpu()
for review, probability in zip(reviews, probabilities):
    label = reloaded.config.id2label[int(probability.argmax())]
    print(f"{label:8s} p={probability.max().item():.3f} | {review}")


## 10. Things to Try

- Fine-tune only the final Transformer block before unfreezing the complete encoder.
- Repeat with fewer labeled reviews and compare data efficiency.
- Inspect confident errors and negation-heavy examples.
- Replace DistilBERT with a multilingual or domain-specific encoder.

## Troubleshooting

- The first run requires internet access for the dataset and model; later runs use the Hugging Face cache.
- Enable a GPU for normal full fine-tuning. Smoke mode verifies the workflow but does not produce a useful classifier.
- If memory is exhausted, lower the per-device batch size and use gradient accumulation before shortening sequences.
- A new classification head warning is expected: those task-specific weights must be learned.

**References:** [Hugging Face text classification](https://huggingface.co/docs/transformers/tasks/sequence_classification), [DistilBERT](https://arxiv.org/abs/1910.01108), [Rotten Tomatoes dataset](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes).
